# Overview

[![arXiv](https://img.shields.io/badge/Slides-ffc508?style=for-the-badge&logo=google)](https://docs.google.com/presentation/d/1pPUca5xodYpZi4p2X1khqaJjueEx-dZ52y-9xHPcpdU/edit?usp=sharing)
[![GitHub](https://img.shields.io/badge/Repo-060e1a?style=for-the-badge&logo=github)](https://github.com/ATATC/MiniCourse-MLLM)

This is the hands-on tutorial for the mini course on MLLM. In this tutorial, we will cover a complete pipeline from **data preparation**, **fine-tuning**, **inference**, and **evaluation**, using **MedGemma 1.5 4B**.

**[Note]** You would need a GPU runtime with >=40 GB VRAM: A100, H100, or G4.

# Data Preparation

## Step 0: Set up the environment on Colab

### Authentication through environment variables

**[Action Required]** Please define your Hugging Face token as `HF_TOKEN` in the "Secrets" tab on the left and enable Notebook access. You can learn how to get an access token [here](https://huggingface.co/docs/hub/en/security-tokens).

In [ ]:
from google.colab import userdata
from os import environ

environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### Install dependencies

In [ ]:
!mkdir -p /workspace/input /workspace/output /workspace/app
!cd /workspace/app
!git clone https://github.com/ATATC/MedGemma-FLARE-2D
!pip install -e MedGemma-FLARE-2D

Cloning into 'MedGemma-FLARE-2D'...
remote: Enumerating objects: 748, done.
remote: Counting objects: 100% (282/282), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 748 (delta 198), reused 196 (delta 120), pack-reused 466 (from 1)
Receiving objects: 100% (748/748), 1.58 MiB | 24.83 MiB/s, done.
Resolving deltas: 100% (515/515), done.
Obtaining file:///content/MedGemma-FLARE-2D
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/ProjectNeura/Erbium.git to /tmp/pip-install-29chnu6l/erbium_ea8fc9d1c52a407291200c9a013affb6
  Running command git clone --filter=blob:none --quiet https://github.com/ProjectNeura/Erbium.git /tmp/pip-install-29chnu6l/erbium_ea8fc9d1c52a407291200c9a013affb6
  Resolved https://github.com/ProjectNeura/Erbium.git to co

## Step 1: Download FLARE-MLLM-2D

**[Action Required]** You need to gain access to the [dataset](https://huggingface.co/datasets/FLARE-MedFM/FLARE-MLLM-2D) before proceeding.

There are 3 splits in this dataset: **training**, **validation-hidden**, and **validation-public**. However, only the Xray/IU_XRay subdataset contains the report generation task, so we just download what we need to avoid storage overhead.

In [ ]:
from huggingface_hub import snapshot_download

local_dir = "/workspace/input/FLARE-MLLM-2D"
snapshot_download(
    repo_id="FLARE-MedFM/FLARE-MLLM-2D",
    repo_type="dataset",
    local_dir=local_dir,
    local_dir_use_symlinks=False,
    resume_download=True,
    allow_patterns=[
        "training/Xray/IU_XRay/**",
        "validation-public/Xray/IU_XRay/**"
    ]
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

'/workspace/input/FLARE-MLLM-2D'

### Inspect the raw dataset

Once downloaded, let us see what the dataset contains.

In [ ]:
!ls /workspace/input/FLARE-MLLM-2D

training  validation-public


Let us pull out one sample from the validation-public split.



In [ ]:
from json import load

with open("/workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/IU_XRay_all_val.json") as f:
    print(load(f)[0])

{'TaskType': 'Report_Generation', 'Modality': 'X-Ray', 'ImageName': ['imagesVal/IU-XRay_CXR3030_IM-1405-0.png', 'imagesVal/IU-XRay_CXR3030_IM-1405-1.png'], 'Question': 'From this chest X-ray, write the bone-related observations as a radiologist would.', 'Answer': 'No acute bony abnormality.', 'Split': 'validation'}


## Step 2: Preprocess

Since we use Hugging Face's `transformers` as the backend of the pipeline, we want to convert the dataset format into Hugging Face's.

### Create the configuration for Colab

In Step 0, we have set up the environment on Colab following the convention on [Erbium](https://github.com/ProjectNeura/Erbium). Therefore, we can directly use `erbium_config` that assumes the dataset is in "/workspace/input" and all outputs will be written into "/workspace/output".

In [ ]:
from mle.vars import erbium_config
from mle.interfaces import preprocess

config = erbium_config("MiniCourse_MLLM_DataPreparation", "FLARE-MLLM-2D")
config.initialize()
custom_args = dict(
    tasks=["report_generation"],
    allow_missing_images=False,
    include_unanswered=False,
    no_hf_dataset=False
)

### Run preprocessing

In [ ]:
preprocess(config, False, False, **custom_args)

                                                  Available GPUs                                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name (ID)                                        ┃ Total Memory (GB) ┃ Utilization (%) ┃ Memory Utilization (%) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ NVIDIA RTX PRO 6000 Blackwell Server Edition (0) │       95.6        │      0.00       │          0.65          │
└──────────────────────────────────────────────────┴───────────────────┴─────────────────┴────────────────────────┘

Dataset availability: OK; 2 JSON file(s)

Preprocessed dataset availability: Not found: /workspace/output/Preprocessed-FLARE-MLLM-2D

CUDA version: 12.8

Preprocessing FLARE-MLLM-2D from /workspace/input/FLARE-MLLM-2D

...

Loaded 7797 row(s) from /workspace/input/FLARE-MLLM-2D/training/Xray/IU_XRay/IU_XRay_all_train.json: {'train': 
7797}

Loaded 1945 row(s) from /workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/IU_XRay_all_val.json: 
{'validation': 1945}

Saving the dataset (0/1 shards):   0%|          | 0/7797 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1945 [00:00<?, ? examples/s]

Done. Train=7797 validation=1945 hidden=0 testing=0

Let us explore the preprocessed dataset.

In [ ]:
!ls /workspace/output/Preprocessed-FLARE-MLLM-2D

dataset_info.json  hf_dataset  train.jsonl  validation.jsonl


We see that the preprocessing collapses the splits into single `.jsonl` files. Let us open one up and explore.

In [ ]:
from json import loads

data = []
with open("/workspace/output/Preprocessed-FLARE-MLLM-2D/validation.jsonl") as f:
    for line in f:
        data.append(loads(line))
print(data[0])

{'split': 'validation', 'source': 'IU_XRay', 'case_id': 'IU-XRay_CXR3030_IM-1405-0', 'uid': 'validation:IU_XRay:IU_XRay_all_val:IU-XRay_CXR3030_IM-1405-0:report_generation:0', 'volume_path': '/workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/imagesVal/IU-XRay_CXR3030_IM-1405-0.png', 'image_path': '/workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/imagesVal/IU-XRay_CXR3030_IM-1405-0.png', 'image': '/workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/imagesVal/IU-XRay_CXR3030_IM-1405-0.png', 'images': ['/workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/imagesVal/IU-XRay_CXR3030_IM-1405-0.png', '/workspace/input/FLARE-MLLM-2D/validation-public/Xray/IU_XRay/imagesVal/IU-XRay_CXR3030_IM-1405-1.png'], 'modality': 'X-Ray', 'task_type': 'report_generation', 'metric': 'green_score', 'subtask': 'Report_Generation', 'qid': 0, 'follow_up': '', 'question_type': '', 'prompt': 'Generate a concise diagnostic report for this X-Ray image.\nQuestion: From thi

We can see that the preprocessing also applies a prompt template. This is very important because MedGemma is sensitive to the prompts.

# Fine-tuning

The previous section converted FLARE-MLLM-2D into supervised fine-tuning (SFT) records. Fine-tuning now teaches the model to map the same input structure it will see at inference time, an image plus the MedGemma prompt template, to the expected assistant response.

In this mini course we are not training a multimodal model from scratch. We start from `google/medgemma-1.5-4b-it`, keep most of the pretrained model fixed, and train a small number of adapter weights with LoRA. This is the practical pattern used when the base model is already strong, but we want it to follow a domain-specific data format, vocabulary, or reporting style.

## What the training loop optimizes

Each preprocessed row should contain three ideas:

- **Visual input**: the image file that MedGemma will encode.
- **Instruction text**: the user prompt, already formatted in the chat template expected by the model.
- **Target text**: the reference answer/report the model should generate.

During SFT, the processor turns the image into visual tensors and the text into tokens. The model then predicts the next token of the target answer. The loss is cross entropy over the answer tokens, usually with prompt tokens masked out so the model is not rewarded for copying the instruction.

The important learning signal is therefore: given this medical image and this exact prompt style, produce an answer like the reference answer. A lower validation loss means the model is becoming better at matching the target distribution, but qualitative examples and downstream evaluation are still necessary because report generation can look fluent while missing clinically relevant details.

## Inspect one SFT example

Before launching training, inspect one row from the preprocessed dataset. This is the fastest way to catch data mistakes: wrong image paths, missing answers, unexpected prompt text, or target leakage.

In [ ]:
from pathlib import Path
from datasets import load_dataset

sft_data_dir = Path(config.output_dir) / "Preprocessed-FLARE-MLLM-2D"
train_jsonl = sft_data_dir / "train.jsonl"

train_preview = load_dataset("json", data_files=str(train_jsonl), split="train")
example = train_preview[0]

print(f"Rows: {len(train_preview)}")
print(f"Columns: {list(example.keys())}\n")

for key, value in example.items():
    if isinstance(value, str):
        preview = value[:700].replace("\n", "\\n")
        suffix = "..." if len(value) > 700 else ""
        print(f"{key}: {preview}{suffix}\n")
    else:
        print(f"{key}: {value}\n")

Generating train split: 0 examples [00:00, ? examples/s]

Rows: 7797
Columns: ['split', 'source', 'case_id', 'uid', 'volume_path', 'image_path', 'image', 'images', 'modality', 'task_type', 'metric', 'subtask', 'qid', 'follow_up', 'question_type', 'prompt', 'question', 'answer', 'choices', 'raw_answer', 'messages']

split: train

source: IU_XRay

case_id: IU-XRay_CXR2384_IM-0942-0

uid: train:IU_XRay:IU_XRay_all_train:IU-XRay_CXR2384_IM-0942-0:report_generation:0

volume_path: /workspace/input/FLARE-MLLM-2D/training/Xray/IU_XRay/imagesTr/IU-XRay_CXR2384_IM-0942-0.png

image_path: /workspace/input/FLARE-MLLM-2D/training/Xray/IU_XRay/imagesTr/IU-XRay_CXR2384_IM-0942-0.png

image: /workspace/input/FLARE-MLLM-2D/training/Xray/IU_XRay/imagesTr/IU-XRay_CXR2384_IM-0942-0.png

images: ['/workspace/input/FLARE-MLLM-2D/training/Xray/IU_XRay/imagesTr/IU-XRay_CXR2384_IM-0942-0.png', '/workspace/input/FLARE-MLLM-2D/training/Xray/IU_XRay/imagesTr/IU-XRay_CXR2384_IM-0942-1.png']

modality: X-Ray

task_type: report_generation

metric: green_score

subtask: Re

## Why LoRA and 4-bit loading?

Full fine-tuning updates every model weight. That is expensive for a 4B-parameter multimodal model and easy to overfit on a small course dataset. LoRA instead inserts small trainable low-rank matrices into selected layers. The base model stays frozen, and only these adapters are updated.

`load_in_4bit=True` further reduces memory by loading the frozen base model in 4-bit precision. This combination is often called QLoRA: quantized base weights plus trainable LoRA adapters. The tradeoff is that we get efficient adaptation, not a completely new model. If the base model cannot recognize a finding at all, LoRA on a small dataset may not fix that. If the base model has the knowledge but needs better formatting, task alignment, or domain phrasing, LoRA is usually a good fit.

## Key training choices

| Setting | Value in this notebook | Why it matters |
| --- | --- | --- |
| `image_size` | `896` | Larger images preserve more detail but increase memory and compute. |
| `resize_mode` | `square` | Keeps image preprocessing consistent between training and inference. |
| `max_images_per_sample` | `1` | Keeps the tutorial simple and memory-bounded. Multi-image reports need a different budget. |
| `load_in_4bit` | `True` | Makes the frozen base model fit on a smaller GPU. |
| `lora_rank` | `16` | Controls adapter capacity. Higher rank can learn more but uses more memory and can overfit. |
| `lora_alpha` | `16` | Scales the LoRA update. It is commonly set near the rank for stable first runs. |
| `lora_dropout` | `0.05` | Adds light regularization to the adapters. |
| `gradient_accumulation_steps` | `16` | Simulates a larger batch when the GPU can only hold a small per-device batch. |
| `learning_rate` | `2e-4` | A typical adapter-tuning learning rate. Too high can make outputs unstable. |
| `max_eval_samples` | `256` | Speeds up classroom iteration while still giving validation feedback. |

The main memory levers are `image_size`, per-device batch size, `gradient_accumulation_steps`, and LoRA rank. If training runs out of memory, first reduce per-device batch size or image size. If validation quality is poor but training is stable, try more epochs, a lower learning rate, or a larger LoRA rank.

## Batch size math

The GPU sees the per-device batch size at each forward/backward pass. Gradients are accumulated for several steps before the optimizer updates the adapters. The effective batch size is:

`effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps * number_of_gpus`

For this Colab-style tutorial we assume one GPU, so a per-device batch size of 1 and 16 accumulation steps gives an effective batch size of 16.

For GPUs with >=80 GB VRAM (H100 and G4), set `gradient_accumulation_steps=1`.

In [ ]:
target_effective_batch_size = 16
gradient_accumulation_steps = 2
num_gpus = 1

assert target_effective_batch_size % (gradient_accumulation_steps * num_gpus) == 0
per_device_train_batch_size = target_effective_batch_size // (gradient_accumulation_steps * num_gpus)

print(f"Per-device train batch size: {per_device_train_batch_size}")
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
print(f"Effective batch size: {per_device_train_batch_size * gradient_accumulation_steps * num_gpus}")

Per-device train batch size: 8
Gradient accumulation steps: 2
Effective batch size: 16


## Inspect the training wrapper

We still use the course codebase for the actual training loop because it handles model loading, processor setup, dataset loading, LoRA configuration, checkpointing, and evaluation consistently. However, the wrapper should not be a black box. The next cell prints the function signature and the first part of the implementation so you can connect the notebook arguments to the underlying code.

In [ ]:
import inspect
from mle.interfaces import train

print(inspect.signature(train))

source_lines = inspect.getsource(train).splitlines()
print("\n".join(source_lines[:120]))

(config: mle.vars.ExpConfig, num_epochs: int, batch_size: int, learning_rate: float, use_wandb: bool, smoke_test: bool, **kwargs) -> None
def train(config: ExpConfig, num_epochs: int, batch_size: int, learning_rate: float, use_wandb: bool, smoke_test: bool,
          **kwargs) -> None:
    results = check_environment(config)
    print_environment_check_results(results)
    check_satisfied_or_throw(results, False, True, True)
    _train(config, num_epochs, batch_size, learning_rate, use_wandb, smoke_test, **kwargs)


## Configure the run

The configuration below is intentionally small enough for a hands-on session. For a serious experiment, use more epochs, monitor validation examples, and keep the final checkpoint selected by validation behavior rather than by training loss alone.

In [ ]:
custom_args = dict(
    model_name_or_path="google/medgemma-1.5-4b-it",
    model_output_dir=f"{config.output_dir}/{config.experiment_name}-medgemma15-lora",
    image_size=896,
    resize_mode="square",
    max_images_per_sample=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_eval_samples=256,
    load_in_4bit=True,
    lora_rank=16,
    lora_alpha=16,
    lora_dropout=0.05,
    attn_implementation="auto",
    gradient_checkpointing=True,
    save_steps=200,
    eval_steps=200,
    save_total_limit=3,
    dataloader_num_workers=4,
    seed=42,
    precision="auto"
)

## Step 3: Run fine-tuning

This cell launches one epoch of adapter training. Watch the training loss for optimization problems and the evaluation loss for overfitting. After the run, the adapter checkpoints will be written under `model_output_dir`.

In [ ]:
train(
    config,
    num_epochs=0.25,
    batch_size=per_device_train_batch_size,
    learning_rate=2e-4,
    use_wandb=False,
    smoke_test=False,
    **custom_args
)

                                                  Available GPUs                                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name (ID)                                        ┃ Total Memory (GB) ┃ Utilization (%) ┃ Memory Utilization (%) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ NVIDIA RTX PRO 6000 Blackwell Server Edition (0) │       95.6        │      0.00       │          0.65          │
└──────────────────────────────────────────────────┴───────────────────┴─────────────────┴────────────────────────┘

Dataset availability: OK; 2 JSON file(s)

Preprocessed dataset availability: OK with validation

CUDA version: 12.8

Loading converted FLARE-MLLM-2D data from /workspace/output/Preprocessed-FLARE-MLLM-2D

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Starting MedGemma LoRA fine-tuning

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
122,0.972598,0.694220,0.847115,665215.000000,0.791292


Saved final adapter and processor to /workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora/final

## What to check after training

A finished run is only the start of model selection. Before trusting the adapter, check:

- **Training loss** decreases without sudden spikes or `nan` values.
- **Validation loss** improves or stays stable. If training loss drops while validation loss worsens, the adapter is overfitting.
- **Generated examples** are compared with references, because report quality is not fully captured by loss.
- **Failure modes** are documented by modality and question type. In this dataset, many modalities have zero rows after the current preprocessing run, so the trained adapter mostly reflects the rows that actually loaded.
- **Checkpoint choice** is explicit. Use the checkpoint that behaves best on validation examples, not necessarily the last checkpoint.

# Inference

Fine-tuning produces adapter checkpoints. Inference uses those learned adapter weights to generate answers for new examples. The key idea is that inference must match training: use the same base model, the same image preprocessing assumptions, and the same prompt style.

The `MedGemma-FLARE-2D` codebase already provides an `infer` interface. We will use that interface instead of manually loading `transformers` and `peft` objects in the notebook. This keeps inference aligned with the same code path used by the project scripts while still making the important arguments explicit.

In this section we will:

- Select a trained LoRA checkpoint.
- Inspect the `infer` interface.
- Configure a small validation inference run.
- Generate predictions through `mle.interfaces.infer`.
- Inspect the generated prediction file.

## Choose the checkpoint

The training run saves a final LoRA adapter under `model_output_dir/final` and may also save intermediate `checkpoint-*` directories. We prefer the final adapter when it exists, otherwise we fall back to the latest checkpoint. For a real experiment, choose the checkpoint with the best validation behavior.

In [ ]:
from pathlib import Path

adapter_root = Path(custom_args["model_output_dir"])
final_adapter_path = adapter_root / "final"
checkpoint_dirs = sorted(
    adapter_root.glob("checkpoint-*"),
    key=lambda path: int(path.name.split("-")[-1]) if path.name.split("-")[-1].isdigit() else -1,
)

if (final_adapter_path / "adapter_config.json").exists():
    adapter_path = final_adapter_path
elif checkpoint_dirs:
    adapter_path = checkpoint_dirs[-1]
else:
    adapter_path = final_adapter_path

print(f"Adapter root: {adapter_root}")
print(f"Selected adapter path: {adapter_path}")
print(f"Adapter config exists: {(adapter_path / 'adapter_config.json').exists()}")

if not (adapter_path / "adapter_config.json").exists():
    raise FileNotFoundError(
        "No LoRA adapter checkpoint was found. Run the fine-tuning cell first, "
        "or point adapter_path to a directory that contains adapter_config.json."
    )

Adapter root: /workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora
Selected adapter path: /workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora/final
Adapter config exists: True


## Inspect the inference interface

The public entry point is `mle.interfaces.infer`. It performs the same environment checks as training, then delegates to the inference engine. The important arguments are `tasks`, `use_wandb`, `smoke_test`, and the custom keyword arguments that describe the model, adapter, split, preprocessing, and generation settings.

We inspect it here so the following configuration is connected to the code that will run.

In [ ]:
import inspect
from mle.interfaces import infer

print(inspect.signature(infer))
print("\n".join(inspect.getsource(infer).splitlines()))

(config: mle.vars.ExpConfig, tasks: Sequence[str], use_wandb: bool, smoke_test: bool, **kwargs) -> None
def infer(config: ExpConfig, tasks: Sequence[str], use_wandb: bool, smoke_test: bool, **kwargs) -> None:
    results = check_environment(config)
    print_environment_check_results(results)
    check_satisfied_or_throw(results, False, True, True)
    _infer(config, tasks, use_wandb, smoke_test, **kwargs)


## Configure inference

The inference engine reads the preprocessed split from `config.preprocessed_dataset_dir`, filters rows by task, loads the base model, attaches the adapter if one is provided, generates predictions, and writes a JSONL prediction file.

For the mini course, we run a small validation-public report-generation pass. Increase `max_samples` or change `inference_tasks` when you want a larger run.

In [ ]:
inference_tasks = ["report_generation"]
infer_output_dir = Path(config.output_dir) / f"{config.experiment_name}-medgemma15-lora-infer"

infer_args = dict(
    model_name_or_path=custom_args["model_name_or_path"],
    model_output_dir=custom_args["model_output_dir"],
    adapter_path=str(adapter_path),
    infer_output_dir=str(infer_output_dir),
    predictions_out=str(infer_output_dir),
    split="validation",
    image_size=custom_args["image_size"],
    resize_mode=custom_args["resize_mode"],
    max_images_per_sample=custom_args["max_images_per_sample"],
    batch_size=1,
    max_samples=16,
    max_new_tokens=256,
    temperature=0.0,
    load_in_4bit=True,
    device="auto",
)

## Step 4: Run inference

This cell calls the course interface. Internally, `infer` loads the converted validation rows, filters to `report_generation`, applies the task instruction and prompt, runs generation, and saves predictions under `infer_output_dir`.

We set `temperature=0.0` in `infer_args`, so decoding is deterministic. That is usually the right default for validation and debugging.

In [ ]:
infer(
    config,
    tasks=inference_tasks,
    use_wandb=False,
    smoke_test=False,
    **infer_args,
)

                                                  Available GPUs                                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name (ID)                                        ┃ Total Memory (GB) ┃ Utilization (%) ┃ Memory Utilization (%) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ NVIDIA RTX PRO 6000 Blackwell Server Edition (0) │       95.6        │      0.00       │         83.80          │
└──────────────────────────────────────────────────┴───────────────────┴─────────────────┴────────────────────────┘

Dataset availability: OK; 2 JSON file(s)

Preprocessed dataset availability: OK with validation

CUDA version: 12.8

Loading validation_public rows from /workspace/output/Preprocessed-FLARE-MLLM-2D

...

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generated 16/16 predictions

Saved generated predictions to 
/workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora-infer/validation_public_predictions.jsonl

Saved inference details to 
/workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora-infer/inference_details.json

## Inspect predictions

The interface writes one JSONL prediction file per split. For `split="validation"`, the engine normalizes the split to `validation_public`, so the output file is `validation_public_predictions.jsonl`.

Each record contains the example `uid`, generated `prediction`, normalized `task_type`, and `case_id`. The evaluation section can read this file directly.

In [ ]:
import json
from IPython.display import Markdown, display

prediction_path = infer_output_dir / "validation_public_predictions.jsonl"

with prediction_path.open() as file:
    prediction_records = [json.loads(line) for line in file]

print(f"Loaded {len(prediction_records)} prediction(s) from {prediction_path}")
print(json.dumps(prediction_records[0], indent=2))

display(Markdown("### First model prediction\n" + prediction_records[0]["prediction"]))

Loaded 16 prediction(s) from /workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora-infer/validation_public_predictions.jsonl
{
  "uid": "validation:IU_XRay:IU_XRay_all_val:IU-XRay_CXR3030_IM-1405-0:report_generation:0",
  "prediction": "No acute bony abnormality.",
  "task_type": "report_generation",
  "case_id": "IU-XRay_CXR3030_IM-1405-0"
}


### First model prediction
No acute bony abnormality.

# Evaluation

Inference tells us what the model generated. Evaluation compares those predictions against the reference answers in the preprocessed split and writes metric summaries.

The `MedGemma-FLARE-2D` codebase provides `mle.interfaces.evaluate`, which reads the same converted dataset used by inference, loads the prediction JSONL file, aligns rows by `uid`, computes task-specific metrics, and writes:

- `scores.json`: compact metric summary.
- `details.json`: row/task/split details useful for debugging.

For report generation, the full metric path can use GREEN and CRIMSON. Those scorers can be expensive, so this mini-course section starts with a fast structural check that verifies prediction/reference alignment. To run full report-generation scoring, set `skip_green_score=False` and `skip_crimson_score=False` in `eval_args`.

## Inspect the evaluation interface

The public call mirrors inference: pass the experiment `config`, a list of `tasks`, `use_wandb`, `smoke_test`, and custom keyword arguments for splits, prediction paths, output paths, and metric options.

In [ ]:
import inspect
from mle.interfaces import evaluate

print(inspect.signature(evaluate))
print("\n".join(inspect.getsource(evaluate).splitlines()))

(config: mle.vars.ExpConfig, tasks: Sequence[str], use_wandb: bool, smoke_test: bool, **kwargs) -> None
def evaluate(config: ExpConfig, tasks: Sequence[str], use_wandb: bool, smoke_test: bool, **kwargs) -> None:
    results = check_environment(config)
    print_environment_check_results(results)
    check_satisfied_or_throw(results, False, True, True)
    _evaluate(config, tasks, use_wandb, smoke_test, **kwargs)


## Configure evaluation

`evaluate` must look at the same split and task subset used by `infer`. It also needs to know where the prediction file is. Here, `predictions` points to the inference output directory, so the evaluator will look for `validation_public_predictions.jsonl` inside that directory.

`max_samples` matches the inference run. Without this, evaluation would expect predictions for every selected validation row, while the tutorial inference cell only generated a small batch.

In [ ]:
eval_output_dir = Path(config.output_dir) / f"{config.experiment_name}-medgemma15-lora-eval"
scores_path = eval_output_dir / "scores.json"
details_path = eval_output_dir / "details.json"

eval_args = dict(
    predictions=str(infer_output_dir),
    infer_output_dir=str(infer_output_dir),
    eval_output_dir=str(eval_output_dir),
    split=infer_args["split"],
    max_samples=infer_args["max_samples"],
    iou_threshold=0.5,
    green_batch_size=1,
    green_max_length=2048,
    skip_green_score=False,
    skip_crimson_score=False,
)

## Step 5: Run evaluation

This cell calls the project evaluation interface. It will fail loudly if the prediction file is missing or if predictions cannot be matched to the selected ground-truth rows. That is useful: alignment errors are common in multimodal benchmarks and should be fixed before interpreting metrics.

In [ ]:
evaluate(
    config,
    tasks=inference_tasks,
    use_wandb=False,
    smoke_test=False,
    **eval_args,
)

                                                  Available GPUs                                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name (ID)                                        ┃ Total Memory (GB) ┃ Utilization (%) ┃ Memory Utilization (%) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ NVIDIA RTX PRO 6000 Blackwell Server Edition (0) │       95.6        │      0.00       │          7.74          │
└──────────────────────────────────────────────────┴───────────────────┴─────────────────┴────────────────────────┘

Dataset availability: OK; 2 JSON file(s)

Preprocessed dataset availability: OK with validation

CUDA version: 12.8

Loading validation_public rows from /workspace/output/Preprocessed-FLARE-MLLM-2D

...

Evaluation Summary

Rows: 16

report_generation: green_score = 0.854167 (n=16)

Saved metric summary to /workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora-eval/scores.json

Saved detailed evaluation to /workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora-eval/details.json

## Inspect scores

The score summary is the quick readout. The details file is more useful when debugging because it records the evaluated split, number of rows, task breakdown, and prediction path.

In [ ]:
import json

with scores_path.open() as file:
    scores = json.load(file)

with details_path.open() as file:
    details = json.load(file)

print("scores.json")
print(json.dumps(scores, indent=2))

print("\ndetails.json summary")
print(f"Split: {details.get('split')}")
print(f"Rows evaluated: {details.get('num_rows')}")
print(f"Predictions: {details.get('predictions')}")
print(f"Tasks: {list(details.get('by_task', {}).keys())}")

scores.json
{
  "green_score": 0.8541666666666666,
  "crimson_score": 1.0
}

details.json summary
Split: validation_public
Rows evaluated: 16
Predictions: /workspace/output/MiniCourse_MLLM_DataPreparation-medgemma15-lora-infer/validation_public_predictions.jsonl
Tasks: ['report_generation']
